# Assignment 2 – Parte 1: Scraping de decretos de emergencia 

**Grupo 12 · Temporada 2025 (1 de enero al 31 de mayo).**

**Diseño de la búsqueda.** Usamos el buscador de normas de gob.pe con el término `estado de emergencia` y el filtro avanzado *Normas > Decretos*, mes por mes. No añadimos `precipitaciones` al término ni el filtro de institución PCM: así traemos **todos** los decretos de estado de emergencia y la clasificación por lluvias se hace después con el **título completo** (`es_lluvia`), que es donde aparece el motivo. Esto evita perder decretos cuya sumilla, cortada en la búsqueda, no incluía la palabra "precipitaciones". A cambio, se cuelan normas de otras instituciones, que se quitan por el enlace (`/institucion/pcm/`).

### Imports y configuración

In [1]:
import re
import time
from urllib.parse import quote

import pandas as pd
import requests
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE = "https://www.gob.pe"
TERM = "estado de emergencia"
PAUSA = 2  

MESES = [
    ("2025-01", "01-01-2025", "31-01-2025"),
    ("2025-02", "01-02-2025", "28-02-2025"),
    ("2025-03", "01-03-2025", "31-03-2025"),
    ("2025-04", "01-04-2025", "30-04-2025"),
    ("2025-05", "01-05-2025", "31-05-2025"),
]

## 1. robots.txt
Se descarga e imprime. Explicación:

- `User-agent: *` prohíbe `/admin/` y cualquier URL con `?sheet=` o `&sheet=`
  (es decir, la paginación de los listados).
- `/busquedas` **sí** está permitida: no aparece en ningún `Disallow`.
  Ojo: la paginación (`sheet=2`, `sheet=3`...) cae en el patrón prohibido, por eso
  el script NO agrega `sheet` en la página 1 y solo lo usa si un mes tiene más
  resultados que los que caben en una página.
- A otros bots (GPTBot: 5 s, OAI-SearchBot: 2 s) les piden `Crawl-delay`. Aunque a `*` no
  le fijen uno, indica que el sitio quiere poco tráfico automatizado: hacemos
  pausas >= 2 s para no sobrecargar el servidor ni arriesgar un bloqueo de IP.

### Descargar e imprimir robots.txt

In [2]:
robots = requests.get(
    f"{BASE}/robots.txt", headers={"User-Agent": "Mozilla/5.0"}, timeout=30
).text  # sin User-Agent el sitio devuelve una página de error
print(robots)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



### Funciones

In [3]:
def build_url(desde, hasta, sheet=None):
    url = (
        f"{BASE}/busquedas?contenido=normas&desde={desde}&hasta={hasta}"
        f"&sort_by=recent&term={quote(TERM)}&tipo_norma[]=6-decretos"
    )
    if sheet and sheet > 1:
        url += f"&sheet={sheet}"
    return url


def make_driver():
    opts = Options()
    opts.add_argument("--headless=new") 
    return webdriver.Chrome(options=opts)


def esperar_resultados(driver, timeout=20):
    """Espera a que aparezca el contador 'N Resultados' o un article."""
    WebDriverWait(driver, timeout).until(
        lambda d: d.find_elements(By.TAG_NAME, "article")
        or re.search(r"\d[\d.,]*\s+Resultados?", d.find_element(By.TAG_NAME, "body").text)
    )


def leer_total(driver):
    texto = driver.find_element(By.TAG_NAME, "body").text
    m = re.search(r"(\d[\d.,]*)\s+Resultados?", texto)
    return int(re.sub(r"[.,]", "", m.group(1))) if m else 0


def extraer_articles(driver, mes):
    filas = []
    for art in driver.find_elements(By.TAG_NAME, "article"):
        enlace = art.find_element(By.CSS_SELECTOR, "a.link-principal")
        fecha = art.find_element(By.XPATH, ".//span[starts-with(normalize-space(.), 'Publicado:')]")
        titulo = art.find_element(By.CSS_SELECTOR, "p")
        filas.append(
            {
                "mes": mes,
                "norma": enlace.text.strip(),
                "fecha_publicacion": fecha.text.replace("Publicado:", "").strip(),
                "titulo": titulo.text.strip(),
                "enlace": enlace.get_attribute("href"),
            }
        )
    return filas

### Un solo mes (enero 2025): total de resultados y extracción
Antes de recorrer toda la temporada, se prueba con un solo mes: se abre la búsqueda con Selenium, se espera con `WebDriverWait` a que cargue el contador "N Resultados", se lee el total y se extraen los cuatro datos de cada `<article>`.

In [4]:
driver = make_driver()
try:
    driver.get(build_url("01-01-2025", "31-01-2025"))
    esperar_resultados(driver)          # WebDriverWait, no solo time.sleep()
    total_enero = leer_total(driver)
    filas_enero = extraer_articles(driver, "2025-01")
finally:
    driver.quit()

print("Resultados que muestra la página:", total_enero, "| artículos extraídos:", len(filas_enero))
pd.DataFrame(filas_enero).head(3)

Resultados que muestra la página: 12 | artículos extraídos: 12


,mes,norma,fecha_publicacion,titulo,enlace
0,2025-01,Decreto Supremo N.° 018-2025-PCM,30 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,2025-01,Decreto Supremo N.° 016-2025-PCM,29 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,2025-01,Decreto Supremo N.° 013-2025-PCM,23 de enero de 2025,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...


### Scrapeo mes por mes

In [5]:
driver = make_driver()
filas, resumen = [], []
try:
    for mes, desde, hasta in MESES:
        extraidas, sheet, total = [], 1, None
        while True:
            driver.get(build_url(desde, hasta, sheet))
            try:
                esperar_resultados(driver)
            except TimeoutException:
                break
            if total is None:
                total = leer_total(driver)
            nuevas = extraer_articles(driver, mes)
            extraidas += nuevas
            if not nuevas or len(extraidas) >= total:
                break
            sheet += 1
            time.sleep(PAUSA)
        filas += extraidas
        resumen.append({"mes": mes, "resultados_totales": total or 0, "resultados_extraidos": len(extraidas)})
        time.sleep(PAUSA)
finally:
    driver.quit()

df = pd.DataFrame(filas)

### Verificación

In [6]:
verif = pd.DataFrame(resumen)
verif["coincide"] = verif["resultados_totales"] == verif["resultados_extraidos"]
print(verif.to_string(index=False))
if not verif["coincide"].all():
    print("\n>>> Hay meses donde no coinciden. Revisar antes de seguir.")

    mes  resultados_totales  resultados_extraidos  coincide
2025-01                  12                    12      True
2025-02                   5                     5      True
2025-03                  14                    14      True
2025-04                  17                    17      True
2025-05                  13                    13      True


### Eliminar repetidos
Primero se eliminan los repetidos exactos (mismo enlace). El paso siguiente es quedarse solo con normas de la PCM. Después, ya sin las notas de prensa y otras normas mal escritas, se revisa si queda el mismo decreto con el número escrito distinto (p. ej. `0046-2025-PCM` vs. `046-2025-PCM`).

In [7]:
n0 = len(df)
df = df.drop_duplicates(subset="enlace").reset_index(drop=True)
print(f"Repetidos eliminados (mismo enlace): {n0 - len(df)} (quedan {len(df)})")

Repetidos eliminados (mismo enlace): 0 (quedan 61)


### Quedarse con normas de la PCM
Aunque la búsqueda es de estado de emergencia, se cuelan normas de otras instituciones (por ejemplo, un *Decreto de Alcaldía* de la Municipalidad de San Juan de Miraflores). Nos quedamos solo con las que tienen `/institucion/pcm/` en el enlace.

In [8]:
n0 = len(df)
df = df[df["enlace"].str.contains("/institucion/pcm/")].copy()
print(f"Quitadas por no ser de la PCM (según el enlace): {n0 - len(df)} (quedan {len(df)})")

# número de norma sin ceros de más ("0046-2025-PCM" -> "046-2025-PCM")
df["norma"] = df["norma"].str.replace(r"\b0(\d{3}-\d{4}-)", r"\1", regex=True)

# repetidos por número de norma (mismo decreto con otro enlace)
n0 = len(df)
df = df.drop_duplicates(subset="norma").reset_index(drop=True)
print(f"Repetidos eliminados (mismo número de norma): {n0 - len(df)} (quedan {len(df)})")

# comprobación: todas terminan en -PCM
print("Sufijo -PCM:", df["norma"].str.upper().str.endswith("-PCM").value_counts().to_dict())

Quitadas por no ser de la PCM (según el enlace): 8 (quedan 53)
Repetidos eliminados (mismo número de norma): 0 (quedan 53)
Sufijo -PCM: {True: 53}


### Título completo (entrando a cada enlace)
En la página de búsquedas las sumillas salen **cortadas** (terminan en "..."), y justo se pierde la parte que indica la causa (p. ej. "por impacto de daños a consecuencia de intensas precipitaciones pluviales"). Se probó buscar una API: `busquedas.json` existe, pero devuelve el mismo texto truncado (~220 caracteres), y no hay JSON público por norma. Por eso se entra a **cada enlace** y se lee el título completo con `requests` + `BeautifulSoup` (con `User-Agent` de navegador) y pausa de 1 s entre páginas. Se hace ya sobre las normas únicas y de la PCM, para no pedir páginas que se van a descartar.

In [9]:
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0"}


def leer_titulo_completo(url):
    """Título completo de la página de una norma (o None si no se pudo leer)."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
    except requests.RequestException:
        return None
    caja = BeautifulSoup(r.text, "html.parser").find("div", class_="description")
    if caja is None:
        return None
    # el primer div trae el título; lo demás es texto extra ("DS N° ... PDF ... Descargar")
    primero = caja.find("div")
    return (primero or caja).get_text(" ", strip=True)


titulos_completos = []
for i, url in enumerate(df["enlace"], start=1):
    titulos_completos.append(leer_titulo_completo(url))
    print(f"{i}/{len(df)}", end=" ")
    time.sleep(1)

df["titulo_completo"] = titulos_completos
# si alguna página no se pudo leer, se deja el título corto de la búsqueda
df["titulo_completo"] = df["titulo_completo"].fillna(df["titulo"])
print("\nSin poder leer:", pd.Series(titulos_completos).isna().sum(),
      "| aún terminan en '...':", df["titulo_completo"].str.endswith("...").sum())

1/53 

2/53 

3/53 

4/53 

5/53 

...

50/53 

51/53 

52/53 

53/53 


Sin poder leer: 0 | aún terminan en '...': 0


### Títulos cortados en gob.pe: abrir el PDF
Dos decretos, el **007-2025-PCM** y el **008-2025-PCM** (ambos del 10 de enero), salen con el nombre cortado en **las dos páginas de gob.pe**: tanto en el buscador como en la página propia de la norma el título es solo *"Declaratoria del Estado de Emergencia"*, sin lugar ni motivo. Con ese texto `es_lluvia` no los puede clasificar (el 007 quedaba fuera de las normas de lluvias).

Para identificarlos se tuvo que **abrir el PDF** del decreto (el enlace "Descargar" de la página de la norma) y leer el título oficial de la primera página con `pypdf`. Solo se hace con las normas cuyo título no empieza con "Decreto Supremo", para no descargar PDFs de más. Resultado:
- **007-2025-PCM**: declara el Estado de Emergencia en 19 departamentos y el Callao *por peligro inminente ante intensas precipitaciones pluviales* → **sí es de lluvias**.
- **008-2025-PCM**: declara el Estado de Emergencia en Chiclayo (Lambayeque) *por peligro inminente ante colapso del sistema de alcantarillado* → no es de lluvias.

El título del PDF viene todo en MAYÚSCULAS. Para que se lea como los demás y para que la búsqueda de departamentos (en el título original, con mayúsculas) funcione igual, se pasa a minúsculas y se vuelven a escribir los departamentos tal como están en la lista `departamentos` (p. ej. "Áncash", "La Libertad", "Callao").

In [10]:
from io import BytesIO
from pypdf import PdfReader

departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                 "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]


def leer_titulo_pdf(url):
    """Título oficial desde la primera página del PDF de la norma (o None si no se pudo leer)."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        enlace_pdf = BeautifulSoup(r.text, "html.parser").find("a", href=re.compile(r"\.pdf", re.I))
        if enlace_pdf is None:
            return None
        pdf = requests.get(enlace_pdf["href"], headers=HEADERS, timeout=60)
        pdf.raise_for_status()
    except requests.RequestException:
        return None
    texto = PdfReader(BytesIO(pdf.content)).pages[0].extract_text()
    # el título va desde "DECRETO SUPREMO QUE" hasta "LA PRESIDENTA/EL PRESIDENTE DE LA REPÚBLICA"
    m = re.search(r"DECRETO SUPREMO QUE.*?(?=LA PRESIDENTA|EL PRESIDENTE)", texto, re.S)
    if m is None:
        return None
    # de MAYÚSCULAS a minúsculas, con los departamentos escritos como en la lista
    titulo = " ".join(m.group(0).split()).capitalize()
    for d in departamentos:
        titulo = re.sub(r"\b" + d.lower() + r"\b", d, titulo)
    return titulo


genericos = ~df["titulo_completo"].str.lower().str.startswith("decreto supremo")
print("Normas con título cortado en gob.pe:", genericos.sum())
for i in df.index[genericos]:
    titulo_pdf = leer_titulo_pdf(df.at[i, "enlace"])
    print(f"\n{df.at[i, 'norma']}\n  gob.pe: {df.at[i, 'titulo_completo']}\n  PDF:    {titulo_pdf}")
    if titulo_pdf:
        df.at[i, "titulo_completo"] = titulo_pdf
    time.sleep(1)

Normas con título cortado en gob.pe: 2



Decreto Supremo N.° 008-2025-PCM
  gob.pe: Declaratoria del Estado de Emergencia
  PDF:    Decreto supremo que declara el estado de emergencia en los distritos de chiclayo, josé leonardo ortiz y la victoria de la provincia de chiclayo del departamento de Lambayeque, por peligro inminente ante colapso del sistema de alcantarillado



Decreto Supremo N.° 007-2025-PCM
  gob.pe: Declaratoria del Estado de Emergencia
  PDF:    Decreto supremo que declara el estado de emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, Puno, San Martín, Tacna y la provincia constitucional del Callao, por peligro inminente ante intensas precipitaciones pluviales


### Columnas `es_lluvia`, `tipo` y `motivo`
Se crean con el título completo en minúsculas (`.lower()`) y el operador `in`.

In [11]:
ee = df.copy()
t = ee["titulo_completo"].str.lower()
ee["es_estado_emergencia"] = t.str.contains("estado de emergencia")
ee["es_lluvia"] = t.apply(lambda s: "lluvia" in s or "precipitaciones" in s)
ee["tipo"] = t.apply(lambda s: "prorroga" if ("prórroga" in s or "prorroga" in s)
                     else "declara" if "declara" in s else "otro")
ee["motivo"] = t.apply(lambda s: "peligro inminente" if "peligro inminente" in s
                       else "impacto de daños" if "impacto de daños" in s else "otro")

print(f"Decretos PCM: {len(ee)} | estado de emergencia: {ee['es_estado_emergencia'].sum()} | lluvia: {ee['es_lluvia'].sum()}")
print(ee["tipo"].value_counts().to_dict(), ee["motivo"].value_counts().to_dict())

Decretos PCM: 53 | estado de emergencia: 53 | lluvia: 18
{'prorroga': 37, 'declara': 16} {'otro': 31, 'impacto de daños': 17, 'peligro inminente': 5}


### Revisión de las filas "otro"
**`tipo`:** las 53 normas quedaron como `prorroga` (37) o `declara` (16); ninguna en "otro". Ojo con el 013-2025-PCM: dice "Decreto Supremo que **modifica** el DS 003-2025-PCM, que **prorroga**...". Contiene "prorroga", pero en realidad es una modificación, así que ahí la regla se equivoca.

**`motivo`:** 31 quedaron como "otro". 

**Cruce con `es_lluvia`:** las 31 "otro" son `es_lluvia = False`. De las demás, 18 son lluvia: 16 por "impacto de daños" y 2 por "peligro inminente" (033-2025-PCM y 007-2025-PCM; este último solo se pudo clasificar con el título del PDF). Las que no son lluvia: 032-2025-PCM ("impacto de daños" por **incendio urbano**), 006-2025-PCM (déficit hídrico), 027-2025-PCM y 008-2025-PCM (colapso del alcantarillado; el 008 también con el título del PDF). Es decir, `motivo` no equivale a lluvia; `es_lluvia` es la columna que sirve para eso.

In [12]:
# ¿Se escapó alguna norma de lluvia? Palabras relacionadas en normas con es_lluvia == False
t = ee["titulo_completo"].str.lower()
posibles = t.str.contains("pluvial|inundaci|huaico|deslizamiento|inminente|déficit|deficit") & ~ee["es_lluvia"]
ee.loc[posibles, ["norma", "titulo_completo"]]

,norma,titulo_completo
5,Decreto Supremo N.° 008-2025-PCM,Decreto supremo que declara el estado de emerg...
7,Decreto Supremo N.° 006-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...
26,Decreto Supremo N.° 027-2025-PCM,Decreto Supremo que prorroga el Estado de Emer...


### Solo normas de lluvias

In [13]:
lluvias = ee[ee["es_lluvia"]].copy()
print(f"Normas de lluvias: {len(lluvias)} de {len(ee)} decretos PCM")
lluvias[["norma", "fecha_publicacion", "tipo", "motivo", "titulo_completo"]].head(5)

Normas de lluvias: 18 de 53 decretos PCM


,norma,fecha_publicacion,tipo,motivo,titulo_completo
6,Decreto Supremo N.° 007-2025-PCM,10 de enero de 2025,declara,peligro inminente,Decreto supremo que declara el estado de emerg...
13,Decreto Supremo N.° 024-2025-PCM,26 de febrero de 2025,declara,impacto de daños,Decreto Supremo que declara el Estado de Emerg...
15,Decreto Supremo N.° 021-2025-PCM,19 de febrero de 2025,declara,impacto de daños,Decreto Supremo que declara el Estado de Emerg...
20,Decreto Supremo N.° 034-2025-PCM,12 de marzo de 2025,declara,impacto de daños,Decreto Supremo que declara el Estado de Emerg...
21,Decreto Supremo N.° 033-2025-PCM,12 de marzo de 2025,declara,peligro inminente,Decreto Supremo que declara el Estado de Emerg...


### Departamentos que menciona cada decreto

In [14]:
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
                 "Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
                 "Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
                 "Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
                 "San Martín", "Tacna", "Tumbes", "Ucayali"]


def departamentos_en(titulo):
    """Lista de departamentos mencionados (una vez cada uno), en el título original y por palabra completa."""
    return [d for d in departamentos if re.search(r"\b" + re.escape(d) + r"\b", titulo)]


lluvias["departamentos"] = lluvias["titulo_completo"].apply(departamentos_en)
lluvias["n_departamentos"] = lluvias["departamentos"].str.len()
lluvias[["norma", "tipo", "n_departamentos", "departamentos"]]

,norma,tipo,n_departamentos,departamentos
6,Decreto Supremo N.° 007-2025-PCM,declara,20,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
13,Decreto Supremo N.° 024-2025-PCM,declara,21,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
15,Decreto Supremo N.° 021-2025-PCM,declara,20,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
20,Decreto Supremo N.° 034-2025-PCM,declara,21,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
21,Decreto Supremo N.° 033-2025-PCM,declara,22,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
27,Decreto Supremo N.° 026-2025-PCM,declara,18,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacuch..."
29,Decreto Supremo N.° 057-2025-PCM,declara,1,[Loreto]
30,Decreto Supremo N.° 056-2025-PCM,prorroga,7,"[Áncash, Cusco, Huánuco, La Libertad, Pasco, P..."
32,Decreto Supremo N.° 053-2025-PCM,prorroga,6,"[Amazonas, Áncash, Cusco, Piura, Tumbes, Ucayali]"
33,Decreto Supremo N.° 052-2025-PCM,declara,2,"[Cusco, Madre de Dios]"


**Cómo evitamos contar mal.** Se busca en el título *original* (con mayúsculas) y por palabra completa (`\bDepartamento\b`), no en minúsculas y con `in`. Ejemplo de nuestra temporada: el **DS 038-2025-PCM** menciona Huancavelica pero **no** Ica. Si se busca `"ica" in titulo.lower()`, "ica" aparece dentro de "huanca**velica**" y ese decreto se contaría también para Ica (lo mismo pasa con el **DS 043-2025-PCM**). Con `\bIca\b` solo cuenta cuando aparece la palabra "Ica" sola.

Para el caso de "provincia de Ica del departamento de Ica": `departamentos_en` revisa cada departamento **una sola vez por decreto** (devuelve True/False por departamento, no cuenta las apariciones), así que aunque "Ica" salga dos veces en el título, ese decreto suma 1 para Ica.

In [15]:
# comprobación del ejemplo:
for num in ["038-2025-PCM", "043-2025-PCM"]:
    t = lluvias.loc[lluvias["norma"].str.contains(num), "titulo_completo"].iloc[0]
    ingenuo = [d for d in departamentos if d.lower() in t.lower()]
    print(num, "| cuenta Ica:", "Ica" in ingenuo, "| por palabra completa cuenta Ica:", "Ica" in departamentos_en(t))

038-2025-PCM | cuenta Ica: True | por palabra completa cuenta Ica: False
043-2025-PCM | cuenta Ica: True | por palabra completa cuenta Ica: False


### Decretos por departamento (declaratorias y prórrogas)

In [16]:
filas = []
for d in departamentos:
    tiene = lluvias["departamentos"].apply(lambda lst: d in lst)
    filas.append({
        "departamento": d,
        "declaratorias": int((tiene & (lluvias["tipo"] == "declara")).sum()),
        "prorrogas": int((tiene & (lluvias["tipo"] == "prorroga")).sum()),
    })
por_depto = pd.DataFrame(filas).sort_values(["declaratorias", "prorrogas"], ascending=False).reset_index(drop=True)
por_depto

,departamento,declaratorias,prorrogas
0,Cusco,9,6
1,Áncash,8,6
2,Huánuco,8,5
3,Amazonas,8,4
4,Ayacucho,8,4
5,Lima,8,3
6,Arequipa,8,2
7,Huancavelica,8,2
8,Loreto,8,2
9,Apurímac,7,2


### Guardar archivos de lluvias (carpeta `datos/`)

In [17]:
import os
os.makedirs("datos", exist_ok=True)

salida = lluvias.drop(columns="titulo").rename(columns={"norma": "numero", "fecha_publicacion": "fecha", "titulo_completo": "titulo"})
salida[["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]].to_csv(
    "datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")
por_depto.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
print("Guardados en datos/:", os.listdir("datos"))

Guardados en datos/: ['tabla_final.csv', 'lluvias_por_departamento.csv', 'ubigeos.csv', 'decretos_lluvias.csv', 'decretos_por_departamento.csv']


### Vista previa de las normas de lluvias
Estado final antes de pasar a la Parte 3.

In [18]:
lluvias[["norma", "fecha_publicacion", "tipo", "motivo"]].reset_index(drop=True)

,norma,fecha_publicacion,tipo,motivo
0,Decreto Supremo N.° 007-2025-PCM,10 de enero de 2025,declara,peligro inminente
1,Decreto Supremo N.° 024-2025-PCM,26 de febrero de 2025,declara,impacto de daños
2,Decreto Supremo N.° 021-2025-PCM,19 de febrero de 2025,declara,impacto de daños
3,Decreto Supremo N.° 034-2025-PCM,12 de marzo de 2025,declara,impacto de daños
4,Decreto Supremo N.° 033-2025-PCM,12 de marzo de 2025,declara,peligro inminente
5,Decreto Supremo N.° 026-2025-PCM,5 de marzo de 2025,declara,impacto de daños
6,Decreto Supremo N.° 057-2025-PCM,30 de abril de 2025,declara,impacto de daños
7,Decreto Supremo N.° 056-2025-PCM,30 de abril de 2025,prorroga,impacto de daños
8,Decreto Supremo N.° 053-2025-PCM,24 de abril de 2025,prorroga,impacto de daños
9,Decreto Supremo N.° 052-2025-PCM,23 de abril de 2025,declara,impacto de daños
